In [ ]:
# PhysioNet MI | Kaggle job 0: download + decode all subjects once (CPU only, no GPU needed).
# Settings: Accelerator = None, Internet = On. Run with "Save Version -> Save & Run All".
# Then: open this version -> Output -> "New Dataset", and attach it to jobs 1-10.
import os, shutil, subprocess, sys
from pathlib import Path

BRANCH = 'feature/full-mamba-source-only-physionet109'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/tmp/repo')  # keep the clone out of the dataset output
MNE_DATA = Path('/tmp/mne_data')
CACHE_OUT = Path('/kaggle/working/physionet_cache')


def run(command, cwd=None, env=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    subprocess.run(command, cwd=cwd, env=env, check=True)


run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if not (REPO_PATH / '.git').is_dir():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)
UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1',
     '--index-url', 'https://download.pytorch.org/whl/cpu'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)

MNE_DATA.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED': '1', 'MPLBACKEND': 'Agg', 'MNE_DATA': str(MNE_DATA),
                    'MNE_DATASETS_EEGBCI_PATH': str(MNE_DATA)})
build = (
    "import yaml; from datamodules.physionet import PhysioNetMILOSO; "
    "c=yaml.safe_load(open('configs/full_mamba_source_only.yaml')); "
    "prep=c['preprocessing']['physionet']; "
    f"prep['array_cache_dir']='{CACHE_OUT}'; prep['seed']=0; "
    "dm=PhysioNetMILOSO(prep, subject_id=1); dm.prepare_data(); "
    "arrays=dm._materialize_subject_arrays(); "
    "print('Subjects cached:', len(arrays), '| path:', dm._array_cache_path())"
)
run([PYTHON, '-c', build], cwd=REPO_PATH, env=environment)
for f in CACHE_OUT.glob('*.npz'):
    print(f, f'{f.stat().st_size / 1024**3:.2f} GB')
